# Experiment: Cognitive-driven model

Liao et al., 2024 (arXiv 2404.17520), the best 5 s NGSIM result found (2.85 m). Three streams: a perceived-safety module built on TTC, TET and TIT, driver-behaviour profiling from graph centrality, and a priority-aware module over relative positions; a lightweight transformer ('Leanformer') fuses them, and a mixture decoder predicts. Paper: 0.31 / 0.81 / 1.44 / 2.08 / 2.85 m.

Part of the model comparison in `experiments/MODEL_ALTERNATIVES.md`. Run All; set `DEBUG = False` in section 2 for the real run.

## 0. Decisions & assumptions

Every choice that could change the result is listed here; the cell notes below repeat the ones that apply to each cell.

**Shared by every notebook (S)**

| # | Decision / assumption | Why | Departure from paper / alternative |
|---|---|---|---|
| S1 | Same data for every model: NGSIM us-101 + i-80 (DEBUG: first 250k rows of us-101), scene-level 70/10/20 split, seed 42, train-only normalization. | Differences in the results must come from the model, not the data. | Papers use their own splits, so published numbers are not like-for-like with ours. |
| S2 | Same training budget for every model: config.py defaults (12 epochs, Adam lr 5e-4 decayed x0.6 per epoch, batch 128, weight decay 1e-5, grad clip 1.0). | A fair comparison; per-model tuning would favour whichever model got tuned most. | Each paper's own schedule (e.g. CS-LSTM trains longer with lr 1e-3). Tune later with --mode tune if a model looks under-trained. |
| S3 | Loss = MSE + bivariate-Gaussian NLL + neighbour-goal MSE (`basic_loss`); the risk-scaled loss is used only where this notebook's M table says so (STRAP-R-based variants). Models without a goal head output zero goals, so their goal term is a constant. A model may add its own `aux_loss` (only the diffusion model does). | Keeps one training objective; the risk-scaled loss is STRAP-R's own contribution. | Loss values are therefore not comparable across models: compare RMSE only. |
| S4 | Best checkpoint = lowest validation loss (`fit` default); test RMSE is reported at 1-5 s for that checkpoint. | Same selection rule for every model. | Selecting on val RMSE would be closer to the reported metric. |
| S5 | KNOWN LIMITATION (not fixed): positions are relative to the target at each timestep, so the target's own history is always (0, 0) and lateral speed is always 0 (NGSIM gives longitudinal speed only). The target is seen through speed, acceleration and lane features; lateral context comes from neighbours. | Kept as is (user decision) so every model gets identical inputs. | Papers feed the target's own x/y track; lane-change prediction is handicapped for all models here. |
| S6 | Neighbours = up to 15 risk-selected vehicles (S/O-field > RISK_THRESHOLD), not every nearby vehicle. | Same scene construction as STRAP. | Grid/graph papers use all vehicles in range; ours see only the risk-selected ones. |
| S7 | Output = one bivariate Gaussian per future step (mu_x, mu_y, sigma_x, sigma_y, rho); RMSE uses the mean. Multimodal models are moment-matched to one Gaussian. | The `fit` / `evaluate_trajectory` contract and how STRAP's Table I is scored. | minADE/minFDE over modes would favour multimodal models. |

**Specific to this model (M)**

| # | Decision / assumption | Why | Departure from paper / alternative |
|---|---|---|---|
| M1 | Safety stream: TTC/TET/TIT per vehicle and step (same definitions and TTC* = 3 s as `strap_risk_feats`) -> MLP -> LSTM over the target's and a masked mean of neighbours' values. | The paper's perceived-safety metrics. | The paper's 'risk tendency index' is not defined precisely; left out. |
| M2 | Behaviour stream: degree / closeness / eigenvector centrality + changes (same code as `bat`) for the target -> LSTM. | Paper's graph-centrality driver profiling. | ASSUMPTION: 10 m distance scale, as in `bat`. |
| M3 | Priority stream: per-vehicle LSTM, attention from the target over real vehicles. | Paper's priority-aware module. | - |
| M4 | Leanformer: the three stream vectors are 3 tokens through one small transformer layer (d = 64, 2 heads, ff 128), mean-pooled. | A 'lightweight transformer' fusion. | ASSUMPTION: Leanformer's internals are not public. |
| M5 | 6 latent modes, moment-matched; no manoeuvre labels. | The paper's GMM decoder is not label-supervised in its description. | - |

## 1. Setup

**Decisions in this cell:** seed fixed to `SEED` (42) in every notebook so initialisation is not a source of differences (S1).

In [ ]:
import sys
from pathlib import Path

# Repo root = first folder upwards that contains src/ (works from experiments/ or the root).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import time
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.config import (
    BATCH_SIZE, BETA_LOSS, DEBUG_MAX_ROWS, DEVICE, DROPOUT, DT, LEARNING_RATE, LR_DECAY, N_V,
    NGSIM_LOCATIONS, NUM_EPOCHS, NORMALIZATION_FILE, SEED, STATE_DIM, TF_STEPS, TH_STEPS, WEIGHT_DECAY,
)
from src.data.loader import NGSIMDataLoader, create_dataloader
from src.data.normalization import load_stats, normalize_samples
from src.evaluation.evaluate import evaluate_trajectory
from src.models.strap import STRAP
from src.pipeline import collision_scores, metric_rows, test_groups
from src.train import fit, prepare_data
from experiments.benchmark import compare, save_result

torch.manual_seed(SEED)
print("device:", DEVICE)

## 2. Experiment config

**Decisions in this cell:** training budget is config.py's (S2); `use_risk_loss` as in the M table; collisions scored at 2 m like `--mode collisions`.

In [ ]:
EXP_NAME = "cognitive"          # unique per notebook -> experiments/results/<EXP_NAME>.json
DESCRIPTION = "Cognitive-driven: safety (TTC/TET/TIT) + behaviour centrality + priority attention, Leanformer fusion"

DEBUG = True                  # True: DEBUG_MAX_ROWS of us-101 for a quick check; False: full data
LOCATIONS = ["us-101"] if DEBUG else NGSIM_LOCATIONS

HPARAMS = dict(               # keyword args of src.train.fit
    num_epochs=NUM_EPOCHS,
    use_risk_loss=False,       # False = basic loss (STRAP-B style)
    beta=BETA_LOSS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    lr_decay=LR_DECAY,
    weight_decay=WEIGHT_DECAY,
)

RUN_COLLISIONS = True         # per-pair collision scoring on the test split (slow on full data)
COLLISION_THRESHOLD = 2.0     # metres, GT collision label distance

CKPT_DIR = ROOT / "experiments" / "checkpoints" / EXP_NAME

## 3. Data
Train-only normalization stats and the intention codebook are written to `CKPT_DIR`.

**Decisions in this cell:** same loader, split and train-only normalization as every other notebook (S1, S6); `norm_stats` is loaded here so the model (next section) can recover metres and lane numbers.

In [ ]:
loader = NGSIMDataLoader(locations=LOCATIONS, max_rows=DEBUG_MAX_ROWS if DEBUG else None)
train_s, val_s, centers = prepare_data(loader, save_dir=str(CKPT_DIR))
_, _, test_raw = loader.get_splits()   # cached; raw (unnormalized) test split
# Train-only mean/std; models that need metres or lane numbers undo the normalization with it.
norm_stats = load_stats(str(CKPT_DIR / NORMALIZATION_FILE))
print(f"train/val/test = {len(train_s)}/{len(val_s)}/{len(test_raw)}")

## 4. Model

Any `torch.nn.Module` works if `forward(states, mask, dims)` returns a 3-tuple:

| output | shape | meaning |
|---|---|---|
| `traj_dist` | `(B, T_f=50, 5)` | `[mu_x, mu_y, sigma_x, sigma_y, rho]` of the target, metres relative to its last observed position (x longitudinal, y lateral); `sigma > 0`, `rho` in (-1, 1) |
| `goals` | `(B, N-1, 4)` | neighbour goals; a model without a goal head returns `states.new_zeros(B, N - 1, 4)` |
| third | anything | ignored by training/eval (STRAP returns its risk field) |

Inputs: `states (B, T_h=30, N, 11)` normalized (vehicle 0 = target, rest neighbours, zero-padded),
`mask (B, N)` True = real vehicle, `dims (B, N, 2)` raw length/width in metres.
Feature index: 0 rel-longitudinal m, 1 rel-lateral m, 2 speed/30, 3 lateral speed (always 0), 4 accel/10,
5 type/2, 6 relative lane (target - vehicle, left = +), 7 length/10, 8 width/3, 9 S-field risk, 10 O-field risk.
Raw value = `states * std + mean` with `norm_stats`. STRAP's k-means codebook is only loaded if the model has a `risk_decoder`.

**Decisions in this cell:** three streams (M1-M3), Leanformer fusion (M4), latent mixture decoder (M5). Built after the data cell because it needs `norm_stats`.

In [ ]:
def moment_match(raw, weights):
    """(B, M, T, 5) raw mode outputs + (B, M) mode weights -> (B, T, 5) one Gaussian (S7)."""
    mu = raw[..., :2]
    sig = F.softplus(raw[..., 2:4]) + 1e-3
    rho = torch.tanh(raw[..., 4])
    w = weights[:, :, None]                                         # (B, M, 1)
    mean = (w[..., None] * mu).sum(1)                               # (B, T, 2)
    d = mu - mean[:, None]
    var = (w[..., None] * (sig ** 2 + d ** 2)).sum(1)
    cov = (w * (rho * sig[..., 0] * sig[..., 1] + d[..., 0] * d[..., 1])).sum(1)
    sx, sy = var[..., 0].sqrt(), var[..., 1].sqrt()
    return torch.stack([mean[..., 0], mean[..., 1], sx, sy, (cov / (sx * sy)).clamp(-0.999, 0.999)], dim=-1)

class VehicleEncoder(nn.Module):
    """Shared per-vehicle encoder: FC + leaky ReLU + LSTM over the 30 history steps."""
    def __init__(self, input_dim=STATE_DIM, emb=32, d=64):
        super().__init__()
        self.embed = nn.Linear(input_dim, emb)
        self.lstm = nn.LSTM(emb, d, batch_first=True)

    def forward(self, x):                                           # x (B, T, N, F)
        B, T, N, D_in = x.shape
        seq, (h, _) = self.lstm(F.leaky_relu(self.embed(x.transpose(1, 2).reshape(B * N, T, D_in)), 0.1))
        return seq.reshape(B, N, T, -1), h[-1].reshape(B, N, -1)   # per-step and last-step encodings

TTC_STAR, TTC_CAP = 3.0, 10.0


def safety_features(raw, mask):
    """TTC, TET, TIT per vehicle and step (B, T, N, 3) from denormalized states, same-lane conflicts only."""
    offset, lane = raw[..., 0], torch.round(raw[..., 6])
    speed, length = raw[..., 2] * 30.0, raw[..., 7] * 10.0
    gap = offset.abs() - (length + length[..., :1]) / 2
    closing = torch.where(offset > 0, speed[..., :1] - speed, speed - speed[..., :1])
    same_lane = (lane == 0) & mask[:, None, :]
    same_lane[..., 0] = False
    conflict = same_lane & (gap > 0) & (closing > 0)
    ttc = torch.where(conflict, gap / closing.clamp(min=1e-3), torch.full_like(gap, TTC_CAP)).clamp(max=TTC_CAP)
    tet = torch.cumsum((ttc < TTC_STAR).float(), dim=1) * DT
    tit = torch.cumsum((TTC_STAR - ttc).clamp(min=0), dim=1) * DT
    return torch.stack([ttc / TTC_CAP, tet / TTC_STAR, tit / TTC_STAR], dim=-1)

def behaviour_features(pos, mask, scale=10.0, iters=20):
    """Per-step centralities (degree, closeness, eigenvector) + their changes -> (B, T, N, 6)."""
    N = pos.shape[2]
    d = torch.cdist(pos, pos)
    pair = (mask[:, None, :, None] & mask[:, None, None, :]).float() * (1 - torch.eye(N, device=pos.device))
    W = torch.exp(-d / scale) * pair
    n = mask.sum(-1).float()[:, None, None]
    degree = W.sum(-1) / (n - 1).clamp(min=1)
    dist_sum = (d * pair).sum(-1)
    closeness = torch.where(dist_sum > 0, (n - 1) * scale / dist_sum.clamp(min=1e-6), torch.zeros_like(dist_sum))
    eig = mask[:, None].float().expand_as(degree).clone()
    for _ in range(iters):
        eig = (W @ eig[..., None]).squeeze(-1)
        eig = eig / eig.norm(dim=-1, keepdim=True).clamp(min=1e-6)
    level = torch.stack([degree, closeness, eig], dim=-1) * mask[:, None, :, None].float()
    change = torch.cat([torch.zeros_like(level[:, :1]), level[:, 1:] - level[:, :-1]], dim=1)
    return torch.cat([level, change], dim=-1)

def decode_modes(decoder, head, ctx, weights, T_f=TF_STEPS):
    """ctx (B, M, C): one context per mode, fed at every step of a shared LSTM decoder;
    the M raw outputs are moment-matched with `weights` (B, M) into one Gaussian (S7)."""
    B, M, _ = ctx.shape
    h, _ = decoder(ctx.reshape(B * M, 1, -1).expand(-1, T_f, -1))
    return moment_match(head(h).reshape(B, M, T_f, 5), weights)

class CognitiveModel(nn.Module):
    def __init__(self, norm_stats, d=64, heads=4, dec=128, n_modes=6):
        super().__init__()
        self.register_buffer("mean", torch.tensor(norm_stats["mean"]))
        self.register_buffer("std", torch.tensor(norm_stats["std"]))
        self.M = n_modes
        self.safety_mlp = nn.Linear(6, 32)
        self.safety_lstm = nn.LSTM(32, d, batch_first=True)
        self.beh_lstm = nn.LSTM(6, d, batch_first=True)
        self.encoder = VehicleEncoder(STATE_DIM, 32, d)
        self.priority = nn.MultiheadAttention(d, heads, batch_first=True)
        self.leanformer = nn.TransformerEncoderLayer(d, 2, 2 * d, dropout=DROPOUT, batch_first=True)   # M4
        self.mode_logits = nn.Linear(d, n_modes)
        self.mode_embed = nn.Embedding(n_modes, 16)
        self.decoder = nn.LSTM(d + 16, dec, batch_first=True)
        self.head = nn.Linear(dec, 5)

    def forward(self, states, mask, dims=None):
        B, _, N, _ = states.shape
        raw = states * self.std + self.mean
        safe = safety_features(raw, mask)                           # M1: (B, T, N, 3)
        nbr = mask.clone()
        nbr[:, 0] = False
        w = nbr[:, None, :, None].float()
        nbr_safe = (safe * w).sum(2) / w.sum(2).clamp(min=1)
        _, (s, _) = self.safety_lstm(F.relu(self.safety_mlp(torch.cat([safe[:, :, 0], nbr_safe], -1))))
        _, (b, _) = self.beh_lstm(behaviour_features(raw[..., :2], mask)[:, :, 0])   # M2: target's profile
        _, h = self.encoder(states)
        p, _ = self.priority(h[:, :1], h, h, key_padding_mask=~mask)  # M3
        tokens = torch.stack([s[-1], b[-1], p[:, 0]], 1)
        ctx = self.leanformer(tokens).mean(1)
        mode_ctx = torch.cat([ctx[:, None].expand(-1, self.M, -1), self.mode_embed.weight[None].expand(B, -1, -1)], -1)
        traj = decode_modes(self.decoder, self.head, mode_ctx, torch.softmax(self.mode_logits(ctx), -1))
        return traj, states.new_zeros(B, N - 1, 4), None


def build_model():
    return CognitiveModel(norm_stats)

model = build_model().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"{type(model).__name__}: {n_params:,} parameters")

## 5. Train

**Decisions in this cell:** `fit` with the config budget (S2); best checkpoint by val loss (S4). Loss curves are only comparable within this notebook (S3).

In [ ]:
t0 = time.time()
training_log = fit(model, train_s, val_s, centers, save_dir=str(CKPT_DIR), **HPARAMS)
train_minutes = (time.time() - t0) / 60

log_df = pd.DataFrame(training_log)
display(log_df)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
log_df.plot(x="epoch", y=["train_loss", "val_loss"], marker=".", ax=axes[0], title="Loss")
log_df.plot(x="epoch", y="val_rmse", marker=".", ax=axes[1], title="Val RMSE (m)")
plt.tight_layout(); plt.show()

## 6. Evaluate (best checkpoint) — RMSE 1s…5s overall, per site, zero-neighbour scenes

**Decisions in this cell:** RMSE of the predicted mean at 1-5 s on the test split, overall / per site / target-only scenes (S7).

In [ ]:
model.load_state_dict(torch.load(CKPT_DIR / "best_model.pt", map_location=DEVICE, weights_only=True))
model.eval()
test_s = normalize_samples(test_raw, norm_stats)

test_metrics = {}
for label, samples in test_groups(test_s).items():
    dl = create_dataloader(samples, batch_size=32, shuffle=False)
    test_metrics[label] = {**evaluate_trajectory(model, dl, DEVICE), "n": len(samples)}
display(pd.DataFrame(test_metrics).T)

## 7. Collision prediction (optional)
Model's predicted target vs constant-velocity neighbours, every risk metric at its default threshold.

**Decisions in this cell:** neighbours are rolled forward at constant velocity and candidates are the nearest vehicles at the last history step, as in `--mode collisions`; only the target trajectory comes from this model.

In [ ]:
collisions = None
if RUN_COLLISIONS:
    scores, gt, _ = collision_scores(model, loader, test_raw, norm_stats, COLLISION_THRESHOLD)
    collisions = {"threshold": COLLISION_THRESHOLD, "n_pairs": len(gt),
                  "n_gt_collisions": int(gt.sum()), "rows": metric_rows(scores, gt)}
    print(f"{len(gt)} pairs, {int(gt.sum())} GT collisions")
    display(pd.DataFrame(collisions["rows"]).set_index("metric"))

## 8. Save result

**Decisions in this cell:** one JSON per `EXP_NAME`; a rerun overwrites it.

In [ ]:
save_result({
    "name": EXP_NAME,
    "description": DESCRIPTION,
    "model_class": type(model).__name__,
    "n_params": n_params,
    "hparams": HPARAMS,
    "data": {"locations": LOCATIONS, "debug": DEBUG},
    "train_minutes": train_minutes,
    "training_log": training_log,
    "test": test_metrics,
    "collisions": collisions,
})

## 9. Comparison & benchmarking — all results in `experiments/results/`

Ranked by overall test `rmse_avg` (best per column highlighted). Check the `data` column:
debug and full-data runs are listed together but are not comparable.
Also written to `experiments/comparison_rmse.csv` / `comparison_collisions.csv`.

**Decisions in this cell:** compares every saved result; only rows with the same `data` value are like-for-like.

In [ ]:
rmse_df, collision_df = compare()